# House Stage 1: final ensemble, validated on the 2024 holdout

The provisional Stage 1 specification, **frozen before 2024 is scored**, then tested once on the holdout.

**Model A, polling-informed:**
$$\hat M_A=\beta_0+\beta_1P+\beta_2G+2.5\,I+\beta_3(P\cdot I)+\beta_4\,\Delta G_{30}$$
`partisan_lean` (P), `gen_margin_raw` (G), `inc_signed` (I, coefficient **fixed at +2.5**), `partisan_lean:inc_signed`, and `gb_momentum_30d_raw` (ΔG₃₀, the generic-ballot change over the previous 30 days).

**Model B, fundamentals:**
$$\hat M_B=\alpha_0+\alpha_1P+\alpha_2F+\alpha_3(S\cdot T)+\alpha_4(S\cdot\pi)$$
`partisan_lean` (P), `fundraising_advantage` (F, log D − log R receipts), `pres_party_signed:is_midterm` (S·T, the president's party's midterm penalty), and `pres_party_signed:cpi_region_log_yoy` (S·π, regional inflation, signed). No generic ballot, incumbency, gasoline, college or Hispanic share (asserted below).

**Final ensemble:**
$$\hat M_{\mathrm{final}}=0.85\,\hat M_A+0.15\,\hat M_B$$

**Estimation:** both models by **equal-year WLS**. Each valid training race is weighted ∝ 1 / n_y, where n_y is the number of valid training races in its election year (`run_wls(..., equal_year=True)`), so every training year counts the same.

**Signs:**

- `inc_signed`: +1 for a Democratic incumbent, −1 for a Republican one, 0 for an open seat.
- `pres_party_signed`: +1 for a Democratic president, −1 for a Republican one.

**The holdout:** 2024 has been reserved in every other notebook (`HOLDOUT_YEARS`). Here `USE_HOLDOUT = True`, and the models are fit on 2006–2022 and scored once on 2024.

- **A fair test:** the 0.15 weight and every specification choice were made on the 2018–2022 backtest, never on 2024.
- **One look only:** after this run, 2024 is no longer a clean holdout, so nothing should be retuned on its result.

The 2018–2022 rolling backtest of the same frozen specification comes first, as the reference. Every model in a window uses the same rows. The evaluation subsets are the backtest's: valid, `competitive_ex_ante` (|lean + generic ballot| ≤ 10, evaluation only) and `competitive_ex_post` (|margin| ≤ 7.5).

**Caveat:** `fundraising_advantage`'s historical totals run past Election Day, so Model B's contribution is somewhat optimistic for a truly pre-election forecast.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import Markdown
from patsy import ModelDesc

# The repository root is the first folder up from here that holds pyproject.toml, so this works at any depth
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))  # repository root, for experiments.house
import house.regression as hr  # hr.<Tab> lists everything; names are in src/house/regression/__init__.py
from experiments.house import ensemble, evaluation, tracking
from experiments.house.run import run_wls

In [2]:
import warnings

with warnings.catch_warnings(action="ignore"):  # loading warns with the same summaries printed below
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, every year (2006-2026)
    targets = hr.load_targets()        # data/processed/house/targets.csv, every year (2006-2024)
print(hr.validate_schema(predictors).summary())  # .issues lists every affected key

## The frozen specification

In [3]:
TARGET = "adj_margin_2p_norm"
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
EX_ANTE_THRESHOLD = 10
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),  # evaluation only
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),
}
SECTIONS = {"EX_ANTE": "competitive_ex_ante", "EX_POST": "competitive_ex_post", "ALL": "valid"}

FIXED_INC = 2.5         # Model A's imposed incumbency coefficient
WEIGHT_B = 0.15         # the final ensemble: (1 - WEIGHT_B) * A + WEIGHT_B * B
EQUAL_YEAR_WEIGHTS = True

# Known before every election: the White House party (+1 D, -1 R) and whether the election is a midterm
PRES_PARTY_SIGNED = {2006: -1, 2008: -1, 2010: 1, 2012: 1, 2014: 1, 2016: 1, 2018: -1, 2020: -1, 2022: 1, 2024: 1,
                     2026: -1}
predictors = predictors.assign(pres_party_signed=predictors["year"].map(PRES_PARTY_SIGNED),
                               is_midterm=(predictors["year"] % 4 == 2).astype(int))

MODELS = {  # name: (formula, fixed coefficients)
    "A_polling": (f"{TARGET} ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed"
                  " + gb_momentum_30d_raw", {"inc_signed": FIXED_INC}),
    "B_fundamentals": (f"{TARGET} ~ partisan_lean + fundraising_advantage + pres_party_signed:is_midterm"
                       " + pres_party_signed:cpi_region_log_yoy", {}),
}
A, B = MODELS

# Guards on Model B: no generic ballot, no incumbency / gasoline / college / Hispanic, national terms only signed
_, b_columns = evaluation.formula_variables(MODELS[B][0], predictors.columns)
GB_COLUMNS = ["gen_margin_raw", *(c for c in predictors.columns if c.startswith("gb_"))]
EXCLUDED = ["inc_signed", "log_gas_yoy", "college", "rel_state_college", "rel_college", "hispanic", "rel_state_hispanic"]
assert not set(b_columns) & set(GB_COLUMNS + EXCLUDED), b_columns
b_terms = [term.name() for term in ModelDesc.from_formula(MODELS[B][0]).rhs_termlist]
assert {"pres_party_signed:is_midterm", "pres_party_signed:cpi_region_log_yoy"} <= set(b_terms)
assert not {"pres_party_signed", "is_midterm", "cpi_region_log_yoy"} & set(b_terms), b_terms

SHARED_COLUMNS = evaluation.shared_columns(formulas=[formula for formula, _ in MODELS.values()],
                                           columns=predictors.columns)
for name, (formula, fixed) in MODELS.items():
    print(f"{name:16s} {formula}   fixed: {fixed or 'none'}")
print(f"final = {1 - WEIGHT_B:.2f} * {A} + {WEIGHT_B:.2f} * {B}; shared columns: {SHARED_COLUMNS}")

A_polling        adj_margin_2p_norm ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed + gb_momentum_30d_raw   fixed: {'inc_signed': 2.5}
B_fundamentals   adj_margin_2p_norm ~ partisan_lean + fundraising_advantage + pres_party_signed:is_midterm + pres_party_signed:cpi_region_log_yoy   fixed: none
final = 0.85 * A_polling + 0.15 * B_fundamentals; shared columns: ['partisan_lean', 'gen_margin_raw', 'inc_signed', 'gb_momentum_30d_raw', 'fundraising_advantage', 'pres_party_signed', 'is_midterm', 'cpi_region_log_yoy']


In [4]:
def fit(name, train_years, eval_year, use_holdout):
    formula, fixed = MODELS[name]
    return run_wls(
        formula=formula,
        train_years=train_years,
        eval_years=[eval_year],
        predict_years=[],
        equal_year=EQUAL_YEAR_WEIGHTS,
        fixed=fixed,
        predictors=predictors,
        targets=targets,
        use_holdout=use_holdout,
        require_complete=SHARED_COLUMNS,
        race_types=EVAL_RACE_TYPES,
        competitive=COMPETITIVE,
        name=name,
        save=False,
    )


def final_metrics(a, b):
    """(metrics, scored) of the final ensemble of results a and b, scored like a single model."""
    metrics, _, scored = evaluation.evaluate_predictions(ensemble.blend(a, b, WEIGHT_B), targets, predictors, TARGET,
                                                         competitive=COMPETITIVE, race_types=EVAL_RACE_TYPES)
    return metrics, scored


METRICS = ["n", "mae", "rmse", "bias", "accuracy_pct", "precision_D_pct", "precision_R_pct", "missed_races"]
DELTAS = ["mae", "rmse", "accuracy_pct", "missed_races"]


def summary(metrics_by_model):
    """One row per model: METRICS per section, then Δ (model - A) on the same districts."""
    rows = {}
    for name, metrics in metrics_by_model.items():
        row = {}
        for section, subset in SECTIONS.items():
            values, base = metrics.loc[(subset, "all")], metrics_by_model[A].loc[(subset, "all")]
            row |= {(section, m): values[m] for m in METRICS}
            if name != A:
                row |= {(section, f"Δ{m}"): values[m] - base[m] for m in DELTAS}
        rows[name] = row
    return pd.DataFrame.from_dict(rows, orient="index")

## Reference: the 2018–2022 rolling backtest of the frozen specification

Each eval year is fit on the elections before it (2006–2016 for 2018; 2006–2018 for 2020; 2006–2020 for 2022), then averaged with each year weighted equally. 2024 isn't touched here (`use_holdout=False`).

In [5]:
BACKTEST = {2018: list(range(2006, 2018, 2)), 2020: list(range(2006, 2020, 2)), 2022: list(range(2006, 2022, 2))}

with warnings.catch_warnings(action="ignore"):  # FEC-missing districts are excluded with a warning, as in every notebook
    backtest = {year: {name: fit(name, train, year, use_holdout=False) for name in MODELS}
                for year, train in BACKTEST.items()}
by_year = {}
for year, fits in backtest.items():
    assert len({(r.counts["train_rows_hash"], r.counts["eval_rows_hash"]) for r in fits.values()}) == 1, year
    by_year[year] = summary({**{name: r.metrics for name, r in fits.items()},
                             "final": final_metrics(fits[A], fits[B])[0]})
backtest_mean = sum(by_year.values()) / len(by_year)
display(Markdown("### Backtest mean (2018, 2020, 2022)"))
tracking.style_best_of(backtest_mean)

### Backtest mean (2018, 2020, 2022)

## The 2024 holdout

All three models fit once on 2006–2022 (equal-year WLS) and scored on 2024. Δ is model − A on the same districts: ✓ marks an improvement and ✗ a regression. `coverage` lists 2024's districts by exclusion reason.

In [6]:
USE_HOLDOUT = True  # the one deliberate use of HOLDOUT_YEARS (2024)
HOLDOUT_TRAIN = list(range(2006, 2024, 2))  # 2006-2022
HOLDOUT_YEAR = 2024

with warnings.catch_warnings(action="ignore"):
    holdout = {name: fit(name, HOLDOUT_TRAIN, HOLDOUT_YEAR, use_holdout=USE_HOLDOUT) for name in MODELS}
assert len({(r.counts["train_rows_hash"], r.counts["eval_rows_hash"]) for r in holdout.values()}) == 1
final_2024, final_scored = final_metrics(holdout[A], holdout[B])
holdout_summary = summary({**{name: r.metrics for name, r in holdout.items()}, "final": final_2024})
display(Markdown(f"### {HOLDOUT_YEAR} holdout (trained on {HOLDOUT_TRAIN[0]}-{HOLDOUT_TRAIN[-1]})"))
display(tracking.style_best_of(holdout_summary))
display(Markdown("### 2024 coverage"))
holdout[A].coverage

### 2024 holdout (trained on 2006-2022)

### 2024 coverage

,n_districts,missing_target,imputed_target,no_prediction,n_valid,competitive_ex_ante,competitive_ex_ante_undetermined,competitive_ex_post,competitive_ex_post_undetermined
year,,,,,,,,,
2024,435,0,37,0,398,80,0,49,0
all,435,0,37,0,398,80,0,49,0


In [7]:
# the final ensemble: 2024 against the backtest mean, side by side
side_by_side = pd.concat({"backtest 2018-2022": backtest_mean.loc["final"], "holdout 2024": holdout_summary.loc["final"]},
                         axis=1).loc[[(s, m) for s in SECTIONS for m in ["mae", "rmse", "bias", "accuracy_pct",
                                                                         "Δmae", "Δaccuracy_pct"]]]
side_by_side.style.format(precision=2)

## 2024 diagnostics

- **Residual correlation:** A's and B's residuals (actual − predicted) on the same districts.
- **Winner flips:** every competitive race where the final ensemble calls a different winner than A.
- **Misses:** every competitive race the final ensemble called wrong.

Margins are D − R, in points.

In [8]:
ensemble.error_correlation([(holdout[A], holdout[B])], list(SECTIONS.values())).style.format(
    {"n": "{:.0f}", "corr": "{:.2f}", "bias_a": "{:+.2f}", "bias_b": "{:+.2f}", "mae_a": "{:.2f}", "mae_b": "{:.2f}"})

,n,corr,bias_a,bias_b,mae_a,mae_b
subset,,,,,,
competitive_ex_ante,80,0.69,-1.36,-4.36,4.56,5.57
competitive_ex_post,49,0.81,-2.01,-5.04,5.60,5.75
valid,398,0.53,-1.03,-3.62,4.68,6.86


In [9]:
a = holdout[A].scored
b = holdout[B].scored[[*evaluation.KEY, "predicted_margin"]].rename(columns={"predicted_margin": "B"})
races = (a.merge(final_scored[[*evaluation.KEY, "predicted_margin", "predicted_winner"]], on=evaluation.KEY,
                 suffixes=("", "_final")).merge(b, on=evaluation.KEY))
races = races[races["competitive_ex_ante"] | races["competitive_ex_post"]].assign(
    district=lambda d: [evaluation.district_label(s, n) for s, n in zip(d["state"], d["district"])],
    A_correct=lambda d: d["predicted_winner"] == d["actual_winner"],
    final_correct=lambda d: d["predicted_winner_final"] == d["actual_winner"])
columns = {"district": "district", "competitive_ex_ante": "ex_ante", "competitive_ex_post": "ex_post",
           "actual_margin": "actual", "predicted_margin": "A", "B": "B", "predicted_margin_final": "final",
           "actual_winner": "winner", "predicted_winner": "A_called", "predicted_winner_final": "final_called"}
margins = {"actual": "{:+.1f}", "A": "{:+.1f}", "B": "{:+.1f}", "final": "{:+.1f}"}

flips = races[races["predicted_winner"] != races["predicted_winner_final"]]
flips = flips[list(columns)].rename(columns=columns).assign(
    verdict=lambda d: ["fixed" if f == w else "broken" for f, w in zip(d["final_called"], d["winner"])])
display(Markdown(f"### 2024 winner flips (final vs A): {int((flips['verdict'] == 'fixed').sum())} fixed, "
                 f"{int((flips['verdict'] == 'broken').sum())} broken"))
display(flips.style.format(margins).map(
    lambda v: f"color: {tracking.GOOD}" if v == "fixed" else f"color: {tracking.BAD}" if v == "broken" else "",
    subset=["verdict"]))

misses = races[~races["final_correct"]][list(columns)].rename(columns=columns).sort_values("actual", key=abs)
display(Markdown(f"### 2024 competitive races the final ensemble missed: {len(misses)}"))
misses.style.format(margins)

### 2024 winner flips (final vs A): 0 fixed, 0 broken

,district,ex_ante,ex_post,actual,A,B,final,winner,A_called,final_called,verdict


### 2024 competitive races the final ensemble missed: 9

,district,ex_ante,ex_post,actual,A,B,final,winner,A_called,final_called
281,OH-09,True,True,+0.7,-1.7,+2.7,-1.0,D,R,R
176,ME-02,False,True,+0.7,-6.2,-2.0,-5.5,D,R,R
72,CO-08,True,True,-0.8,+3.7,+6.7,+4.2,R,D,D
303,PA-07,True,True,-1.0,+0.3,+6.4,+1.2,R,D,D
265,NY-19,True,True,+2.2,-1.8,+3.8,-1.0,D,R,R
381,WA-03,True,True,+3.9,-4.4,+2.8,-3.3,D,R,R
263,NY-17,True,True,-6.4,+5.7,+6.5,+5.8,R,D,D
36,CA-22,False,True,-6.8,+10.2,+9.9,+10.1,R,D,D
184,MI-08,True,True,+6.9,-0.3,+0.6,-0.2,D,R,R


## Coefficients

The 2024 fit (2006–2022, equal-year WLS). In Model A, `inc_signed` is the imposed +2.5, with no standard error.

In [10]:
pd.concat({name: r.coefficients[["coef", "std_err", "p_value"]] for name, r in holdout.items()}, axis=1).round(3)

A_polling                 B_fundamentals  \
                                          coef std_err p_value           coef   
term                                                                            
Intercept                               -1.045   0.471   0.027          2.291   
partisan_lean                            1.033   0.007   0.000          0.678   
gen_margin_raw                           1.306   0.085   0.000            NaN   
partisan_lean:inc_signed                -0.088   0.010   0.000            NaN   
gb_momentum_30d_raw                     -1.680   0.248   0.000            NaN   
inc_signed                               2.500     NaN     NaN            NaN   
fundraising_advantage                      NaN     NaN     NaN          4.724   
pres_party_signed:is_midterm               NaN     NaN     NaN         -3.189   
pres_party_signed:cpi_region_log_yoy       NaN     NaN     NaN         -0.132   

                                                      
                                     std_err p_value  
term                                                  
Intercept                              0.160   0.000  
partisan_lean                          0.009   0.000  
gen_margin_raw                           NaN     NaN  
partisan_lean:inc_signed                 NaN     NaN  
gb_momentum_30d_raw                      NaN     NaN  
inc_signed                               NaN     NaN  
fundraising_advantage                  0.087   0.000  
pres_party_signed:is_midterm           0.276   0.000  
pres_party_signed:cpi_region_log_yoy   0.059   0.026